# ConceptNet-100k → recursive lossless graph dictionary v2 — L4 benchmark

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SemanticMap/semgraphex/blob/feature/recursive-lossless-graph-dictionary-v2/notebooks/14_conceptnet_100k_recursive_grammar_l4_colab.ipynb)

Полный воспроизводимый прогон PR #13: typed-WL/Wishart discovery → persistent graph dictionary → exact transition archives → recursive coarsening → consolidated final→level0 lossless decode → grammar-induced multiscale graph/graphex projection.

**Рекомендуемый runtime:** Colab **L4 GPU + High-RAM**. Все тяжёлые вычисления идут в `/content`; Google Drive используется только как устойчивое хранилище входов, checkpoints и финальных артефактов.

После успешного выполнения итоговый каталог содержит `COMPLETED`, `hierarchy_exact_v1.zip`, `hierarchy_codec_report.json`, `compression_analysis.json` и `multiscale_graph_model.json`.

In [ ]:
# ===== USER PARAMETERS =====
REPO_URL = 'https://github.com/SemanticMap/semgraphex.git'
BRANCH = 'feature/recursive-lossless-graph-dictionary-v2'
PINNED_SHA = 'd70d5133b60d8cd63e8c374501252bf0bc545eac'  # Код эксперимента; не менять между NEW и RESUME.
REPO_ROOT = '/content/semgraphex'

DRIVE_ROOT = '/content/drive/MyDrive/SemanticMap/colab/wishart'
RUN_NAME = 'wishart-grammar-v2-cn100k-l4-01'
RUN_MODE = 'NEW'  # NEW | RESUME

# Предпочтительный устойчивый вход. Если файла нет, ноутбук ищет COMPLETED prepared/100k.
DATASET_DRIVE = 'data/conceptnet_en_100k.tsv'
PREPARED_DRIVE = ''  # Например: prepared/prepare-xxxxxxxxxxxx; оставьте '' для AUTO.

REQUIRE_L4 = True
EXPECTED_NODES = 100000
DEVICE = 'cuda'      # Для этого benchmark намеренно strict CUDA.
CPU_WORKERS = 4
BLAS_THREADS = 1
KEEP_SCRATCH = False

assert RUN_MODE in {'NEW', 'RESUME'}
assert DEVICE == 'cuda', 'Этот notebook предназначен для L4/CUDA benchmark'


In [ ]:
# ===== L4 / RAM / DISK PREFLIGHT =====
import json, os, platform, shutil, subprocess, sys
from pathlib import Path

print('Python:', sys.version.replace('\n', ' '))
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())
disk = shutil.disk_usage('/content')
print('Local /content free GiB:', round(disk.free / 2**30, 2))
try:
    import psutil
    print('RAM total GiB:', round(psutil.virtual_memory().total / 2**30, 2))
except Exception:
    pass

if not shutil.which('nvidia-smi'):
    raise RuntimeError('GPU не обнаружен. Выберите Runtime → Change runtime type → L4 GPU.')
gpu_line = subprocess.check_output([
    'nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'
], text=True).strip()
print('GPU:', gpu_line)
if REQUIRE_L4 and 'L4' not in gpu_line.upper():
    raise RuntimeError(f'Ожидался L4 GPU, получено: {gpu_line}')
if disk.free < 8 * 2**30:
    raise RuntimeError('Для полного 100k run требуется не менее 8 GiB свободного /content scratch.')


In [ ]:
# ===== MOUNT GOOGLE DRIVE =====
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path(DRIVE_ROOT)
DRIVE.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE)


In [ ]:
# ===== RESOLVE EXACT 100k INPUT =====
dataset_path = DRIVE / DATASET_DRIVE
prepared_rel = PREPARED_DRIVE.strip()

if prepared_rel:
    prepared_path = DRIVE / prepared_rel
    if not (prepared_path / 'COMPLETED').is_file():
        raise FileNotFoundError(f'Prepared input is not COMPLETED: {prepared_path}')
    INPUT_KIND, INPUT_REL = 'prepared', prepared_rel
elif dataset_path.is_file():
    INPUT_KIND, INPUT_REL = 'dataset', DATASET_DRIVE
else:
    candidates = []
    prepared_root = DRIVE / 'prepared'
    if prepared_root.is_dir():
        candidates = [p for p in prepared_root.iterdir() if p.is_dir() and (p / 'COMPLETED').is_file()]
    # Prefer prepared inputs that are explicitly 100k when names carry the hint;
    # otherwise use the most recently modified completed prepared directory.
    named_100k = [p for p in candidates if '100k' in p.name.lower()]
    pool = named_100k or candidates
    if not pool:
        raise FileNotFoundError(
            f'Нет {dataset_path} и нет COMPLETED prepared input под {prepared_root}. '
            'Поместите ConceptNet 100k TSV в data/conceptnet_en_100k.tsv или задайте PREPARED_DRIVE.'
        )
    chosen = max(pool, key=lambda p: p.stat().st_mtime)
    INPUT_KIND, INPUT_REL = 'prepared', str(chosen.relative_to(DRIVE))

print({'input_kind': INPUT_KIND, 'input_relative': INPUT_REL})


In [ ]:
# ===== CHECKOUT PINNED CODE + REPRODUCIBLE INSTALL =====
repo = Path(REPO_ROOT)
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', PINNED_SHA], check=True)
sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert sha == PINNED_SHA, (sha, PINNED_SHA)
print('Pinned Git SHA:', sha)

constraints = repo / 'requirements/constraints-colab.txt'
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-c', str(constraints), '-e', f'{repo}[dev,wishart,gpu]'
], check=True)

import torch
assert torch.cuda.is_available(), 'PyTorch CUDA unavailable after install'
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda, 'GPU:', torch.cuda.get_device_name(0))
if REQUIRE_L4 and 'L4' not in torch.cuda.get_device_name(0).upper():
    raise RuntimeError('PyTorch sees a non-L4 GPU; stop this benchmark.')


In [ ]:
# ===== BUILD NEW / RESUME COMMAND =====
config = repo / 'configs/wishart_conceptnet_grammar_v2.yaml'
cmd = [
    sys.executable, '-m', 'semmap_haken.wishart_colab_cli',
    '--config', str(config),
    '--drive-root', DRIVE_ROOT,
    '--scratch-root', '/content/semmap-wishart-grammar-v2',
    '--run-name', RUN_NAME,
    '--device', DEVICE,
    '--cpu-workers', str(CPU_WORKERS),
    '--blas-threads', str(BLAS_THREADS),
    '--compare-mode', 'size_mtime',
]
cmd += ['--prepared-drive', INPUT_REL] if INPUT_KIND == 'prepared' else ['--dataset-drive', INPUT_REL]
if RUN_MODE == 'RESUME':
    cmd += ['--resume']
if KEEP_SCRATCH:
    cmd += ['--keep-scratch']

print('Command:')
print(' '.join(cmd))
print('Durable run:', DRIVE / 'runs' / RUN_NAME)


In [ ]:
# ===== FULL CONCEPTNET-100k / L4 RUN =====
# Runtime interruption is expected to be recoverable: rerun the notebook with
# the same parameters and RUN_MODE='RESUME'.
subprocess.run(cmd, cwd=repo, check=True)


In [ ]:
# ===== EXACTNESS + COMPRESSION + MULTISCALE VALIDATION =====
run = DRIVE / 'runs' / RUN_NAME
assert (run / 'COMPLETED').is_file(), f'Missing COMPLETED: {run}'
input_graph = json.loads((run / 'input_graph_report.json').read_text())
assert int(input_graph['node_count']) == EXPECTED_NODES, input_graph

colab = json.loads((run / 'COLAB_RUN.json').read_text())
hierarchy = json.loads((run / 'hierarchy.json').read_text())
codec = json.loads((run / 'hierarchy_codec_report.json').read_text())
compression = json.loads((run / 'compression_analysis.json').read_text())
multiscale = json.loads((run / 'multiscale_graph_model.json').read_text())

assert codec['roundtrip_exact'] is True
assert compression['summary']['hierarchy_roundtrip_exact'] is True

summary = {
    'run': str(run),
    'git_sha': colab.get('device') and PINNED_SHA,
    'gpu': (colab.get('device') or {}).get('gpu_name'),
    'levels': hierarchy['levels'],
    'nodes': [hierarchy['initial_nodes'], hierarchy['final_nodes']],
    'stop_reason': hierarchy['stop_reason'],
    'dictionary_size': hierarchy.get('dictionary_size'),
    'hierarchy_archive_bytes': compression['summary']['hierarchy_archive_bytes'],
    'binary_bundle_bytes': compression['summary']['hierarchy_binary_bundle_bytes'],
    'hierarchy_ratio_binary': compression['summary']['hierarchy_compression_ratio_binary_bundle'],
    'best_transition_ratio': compression['summary']['best_transition_binary_ratio'],
    'worst_transition_ratio': compression['summary']['worst_transition_binary_ratio'],
    'first_nonwinning_transition': compression['summary']['first_transition_not_beating_binary_baseline'],
    'multiscale_projections': multiscale['projection_count'],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nPer-transition bottlenecks:')
for row in compression['transitions']:
    print({
        'level': row['source_level'],
        'nodes': [row['fine_nodes'], row['coarse_nodes']],
        'ratio_binary': row['compression_ratio_binary'],
        'largest_entry': row['largest_entry'],
        'largest_entry_bytes': row['largest_entry_bytes'],
        'timing_seconds': row.get('timing_seconds'),
    })

print('\nГОТОВО:', run)


## Если Colab оборвался

Не меняйте `PINNED_SHA`, вход, `RUN_NAME` или тип устройства. Поставьте:

```python
RUN_MODE = 'RESUME'
```

и снова выполните notebook сверху вниз. Runner сверит SHA256 входа, YAML, Git revision и CPU/CUDA lineage и восстановится с последнего проверенного level-checkpoint.

## Основная метрика

`hierarchy_ratio_binary < 1` означает, что consolidated recursive grammar archive меньше честного compact exact non-grammar bundle при одинаковом fidelity contract. Сокращение числа узлов само по себе **не** считается доказательством физического сжатия.